# VacinaBR-PNI: ETL via CSVs mensais no Google Drive

Este notebook baixa os ZIPs mensais do Portal de Dados Abertos do SUS, salva em Google Drive e gera a base curada em Parquet, relatorios analiticos e documentacao.

Fonte: https://dadosabertos.saude.gov.br/dataset/doses-aplicadas-pelo-programa-de-nacional-de-imunizacoes-pni-2025

In [ ]:
# Célula 1 - Conecta o Google Colab ao Google Drive.
# Todos os arquivos grandes da pipeline ficam no Drive para persistirem
# depois que a sessão do Colab encerrar.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Célula 2 - Instala a camada analítica local usada pela disciplina/projeto.
# DuckDB funciona como banco SQL embarcado sobre arquivos Parquet, sem servidor.
!pip install -q duckdb

In [ ]:
# Célula 3 - Importa bibliotecas e define a estrutura de pastas no Drive.
# raw/zip guarda os ZIPs originais, processed guarda Parquet curado,
# analytics guarda agregados CSV e docs guarda metadados/documentação.
from __future__ import annotations

import csv
import json
import os
import re
import shutil
import zipfile
from pathlib import Path
from typing import Any

import duckdb
import pandas as pd
import requests

DRIVE_ROOT = Path('/content/drive/MyDrive/VacinaBR-PNI')
RAW_ZIP_DIR = DRIVE_ROOT / 'data' / 'raw' / 'zip'
PROCESSED_DIR = DRIVE_ROOT / 'data' / 'processed'
ANALYTICS_DIR = DRIVE_ROOT / 'data' / 'analytics'
DOCS_DIR = DRIVE_ROOT / 'docs'
DUCKDB_PATH = DRIVE_ROOT / 'data' / 'vacinabr_pni.duckdb'

# CHUNKSIZE controla o uso de memória ao ler CSVs grandes.
# MAX_MONTHS=None deixa o manifesto completo disponível; a célula 7 decide
# quais meses realmente serão processados.
CHUNKSIZE = 100_000
MAX_MONTHS = None
DEDUPLICATE = True
CSV_ENCODING = 'latin1'

# Segurança: esta célula nunca apaga dados gerados.
# Se precisar limpar outputs, faça isso manualmente e de forma explícita no Drive.
# Cria as pastas antes do download/processamento para evitar erros de caminho.
for path in [RAW_ZIP_DIR, PROCESSED_DIR, ANALYTICS_DIR, DOCS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)

In [ ]:
# Célula 4 - Define o manifesto das fontes oficiais.
# Cada item representa um ZIP mensal publicado pelo Portal de Dados Abertos do SUS.
# O manifesto também é salvo em docs/ para registrar exatamente quais arquivos
# alimentaram a versão processada do dataset.
MANIFEST = [
    {'year': 2025, 'month': 1, 'month_name': 'janeiro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_jan_2025_csv.zip'},
    {'year': 2025, 'month': 2, 'month_name': 'fevereiro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_fev_2025_csv.zip'},
    {'year': 2025, 'month': 3, 'month_name': 'marco', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_mar_2025_csv.zip'},
    {'year': 2025, 'month': 4, 'month_name': 'abril', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_abr_2025_csv.zip'},
    {'year': 2025, 'month': 5, 'month_name': 'maio', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_mai_2025_csv.zip'},
    {'year': 2025, 'month': 6, 'month_name': 'junho', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_jun_2025_csv.zip'},
    {'year': 2025, 'month': 7, 'month_name': 'julho', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_jul_2025_csv.zip'},
    {'year': 2025, 'month': 8, 'month_name': 'agosto', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_ago_2025_csv.zip'},
    {'year': 2025, 'month': 9, 'month_name': 'setembro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_set_2025_csv.zip'},
    {'year': 2025, 'month': 10, 'month_name': 'outubro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_out_2025_csv.zip'},
    {'year': 2025, 'month': 11, 'month_name': 'novembro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_nov_2025_csv.zip'},
    {'year': 2025, 'month': 12, 'month_name': 'dezembro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_dez_2025_csv.zip'},
]

# Transforma o manifesto em tabela para inspeção e reprodutibilidade.
manifest_df = pd.DataFrame(MANIFEST)
manifest_df['source_page'] = 'https://dadosabertos.saude.gov.br/dataset/doses-aplicadas-pelo-programa-de-nacional-de-imunizacoes-pni-2025'
manifest_df.to_csv(DOCS_DIR / 'pni_2025_csv_manifest.csv', index=False)
manifest_df

In [ ]:
Q

In [ ]:
# Célula 6 - Baixa os ZIPs mensais e identifica o CSV dentro de cada ZIP.
# Os downloads são salvos em raw/zip no Drive e reutilizados se já existirem,
# evitando baixar novamente arquivos grandes em execuções posteriores.

def download_file(url: str, output_path: Path) -> None:
    # Reaproveita o arquivo se ele já foi baixado com tamanho maior que zero.
    if output_path.exists() and output_path.stat().st_size > 0:
        print('[download] reuse', output_path.name)
        return
    print('[download]', url)

    # Grava primeiro em .part para não deixar um ZIP incompleto com nome final.
    tmp_path = output_path.with_suffix(output_path.suffix + '.part')
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with tmp_path.open('wb') as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
    tmp_path.replace(output_path)

def zip_csv_member(zip_path: Path) -> str:
    # Localiza o primeiro arquivo CSV dentro do ZIP mensal.
    with zipfile.ZipFile(zip_path) as zf:
        csv_names = [name for name in zf.namelist() if name.lower().endswith('.csv')]
        if not csv_names:
            raise RuntimeError(f'Nenhum CSV encontrado em {zip_path}')
        return csv_names[0]

def detect_sep(zip_path: Path, member: str) -> str:
    # Detecta delimitador por contagem simples no cabeçalho: ';' ou ','.
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(member) as file:
            header = file.readline().decode(CSV_ENCODING, errors='replace')
    return ';' if header.count(';') >= header.count(',') else ','

def csv_has_header(zip_path: Path, member: str, sep: str) -> bool:
    # Janeiro/2025 foi publicado sem cabeçalho. Esta função diferencia
    # arquivos com header de arquivos puramente posicionais.
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(member) as file:
            text = file.readline().decode(CSV_ENCODING, errors='replace')
    first_row = next(csv.reader([text], delimiter=sep))
    normalized = {normalize_column_name(value) for value in first_row}
    return bool({'data_vacina', 'codigo_documento', 'tipo_sexo_paciente'} & normalized)

# Seleciona só os primeiros meses durante teste ou todos quando MAX_MONTHS=None.
selected_manifest = MANIFEST[:MAX_MONTHS] if MAX_MONTHS else MANIFEST
for item in selected_manifest:
    filename = Path(item['url']).name
    download_file(item['url'], RAW_ZIP_DIR / filename)

In [ ]:
# Célula 7 - Executa o ETL principal em chunks e salva Parquet particionado.
# Esta versão foi ajustada para retomada: por padrão processa apenas os meses
# restantes, preservando os Parquets já gerados para os meses anteriores.

# Escolha manualmente aqui os meses a processar.
# Exemplos:
#   MONTHS_TO_PROCESS = [10, 11, 12]  # outubro, novembro e dezembro
#   MONTHS_TO_PROCESS = [1]           # apenas janeiro
#   MONTHS_TO_PROCESS = None          # todos os meses do manifesto
MONTHS_TO_PROCESS = [10, 11, 12]

# Deixe False para não apagar partições existentes.
# Use True somente se quiser reprocessar do zero os meses escolhidos acima.
CLEAR_OUTPUTS_FOR_SELECTED_MONTHS = False

# Quando True, meses selecionados que já possuem arquivos part-*.parquet são pulados.
# Isso permite retomar uma execução sem duplicar Parquets já gerados.
SKIP_EXISTING_MONTHS = True

# A deduplicação global mantém um set de hashes em memória e pode estourar RAM
# em execuções longas no Colab. Para processar poucos meses, você pode trocar para True,
# mas o padrão False é mais estável no Colab.
RESUME_DEDUPLICATE = False

resume_manifest = selected_manifest
if MONTHS_TO_PROCESS is not None:
    wanted_months = {int(month) for month in MONTHS_TO_PROCESS}
    resume_manifest = [item for item in selected_manifest if int(item['month']) in wanted_months]

if not resume_manifest:
    raise ValueError('Nenhum mês selecionado para processamento. Confira MONTHS_TO_PROCESS na célula 7.')

print('Meses selecionados:', [item['month'] for item in resume_manifest])

def month_partition_dir(item):
    return PROCESSED_DIR / f"year={item['year']}" / f"month={int(item['month']):02d}"

def month_has_parquet(item):
    month_dir = month_partition_dir(item)
    return month_dir.exists() and any(month_dir.glob('part-*.parquet'))

if SKIP_EXISTING_MONTHS and not CLEAR_OUTPUTS_FOR_SELECTED_MONTHS:
    skipped_months = [int(item['month']) for item in resume_manifest if month_has_parquet(item)]
    resume_manifest = [item for item in resume_manifest if not month_has_parquet(item)]
    if skipped_months:
        print('[skip] meses já existentes com Parquet:', skipped_months)

print('Meses que serão processados agora:', [item['month'] for item in resume_manifest])

if CLEAR_OUTPUTS_FOR_SELECTED_MONTHS:
    for item in resume_manifest:
        month_dir = month_partition_dir(item)
        if month_dir.exists():
            print('[manual] removendo partição selecionada:', month_dir)
            shutil.rmtree(month_dir)

# Métricas desta retomada. A célula 8 recalcula os artefatos finais lendo todos
# os Parquets existentes, incluindo meses já processados em execuções anteriores.
metrics = {
    'original_records': 0,
    'processed_records': 0,
    'removed_duplicate_records': 0,
    'total_missing_values': 0,
    'invalid_age_records': 0,
    'invalid_date_records': 0,
    'complete_records': 0,
    'incomplete_records': 0,
    'valid_document_records': 0,
    'invalid_document_records': 0,
}

missing_metrics = {}
seen_hashes = set()
schema_columns = None

previous_validation_path = DOCS_DIR / 'validation_report.csv'
if previous_validation_path.exists():
    previous_validation = pd.read_csv(previous_validation_path)
    for _, row in previous_validation.iterrows():
        key = str(row['metric'])
        try:
            value = int(row['value'])
        except Exception:
            continue
        if key in metrics:
            metrics[key] = value
        elif key.startswith('missing_'):
            missing_metrics[key] = value
    print('[resume] métricas anteriores carregadas de:', previous_validation_path)
else:
    print('[resume] nenhum validation_report anterior encontrado; métricas começam do zero.')

def update_metrics(df: pd.DataFrame, original_rows: int, removed_duplicates: int) -> None:
    # Atualiza contadores de volume, qualidade, completude e validade documental.
    metrics['original_records'] += original_rows
    metrics['processed_records'] += len(df)
    metrics['removed_duplicate_records'] += removed_duplicates
    metrics['total_missing_values'] += int(df.isna().sum().sum())
    if 'idade_valida' in df.columns:
        metrics['invalid_age_records'] += int((~df['idade_valida'].fillna(False)).sum())
    if 'data_vacina' in df.columns:
        metrics['invalid_date_records'] += int(df['data_vacina'].isna().sum())
    if 'registro_completo' in df.columns:
        metrics['complete_records'] += int(df['registro_completo'].sum())
        metrics['incomplete_records'] += int((~df['registro_completo'].fillna(False)).sum())
    if 'registro_valido_documento' in df.columns:
        metrics['valid_document_records'] += int(df['registro_valido_documento'].sum())
        metrics['invalid_document_records'] += int((~df['registro_valido_documento'].fillna(False)).sum())
    for col in ['data_vacina', 'sexo_paciente', 'numero_idade_paciente', 'uf_paciente', 'codigo_municipio_paciente', 'sg_vacina', 'descricao_vacina_fabricante']:
        if col in df.columns:
            missing_metrics[f'missing_{col}'] = missing_metrics.get(f'missing_{col}', 0) + int(df[col].isna().sum())

def deduplicate_chunk(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    # Em retomadas, a deduplicação global é desativada para reduzir RAM.
    if not RESUME_DEDUPLICATE or df.empty:
        return df, 0
    dedup_subset = [c for c in df.columns if c not in SENSITIVE_COLUMNS]
    hashes = pd.util.hash_pandas_object(df[dedup_subset].astype('string'), index=False).astype('uint64')
    duplicate_mask = hashes.map(lambda value: int(value) in seen_hashes)
    new_hashes = hashes[~duplicate_mask]
    seen_hashes.update(int(value) for value in new_hashes)
    return df.loc[~duplicate_mask].copy(), int(duplicate_mask.sum())

def next_part_counter() -> int:
    # Continua a numeração dos part-* para não sobrescrever partes antigas.
    max_part = -1
    for path in PROCESSED_DIR.rglob('part-*.parquet'):
        match = re.search(r'part-(\d+)\.parquet$', path.name)
        if match:
            max_part = max(max_part, int(match.group(1)))
    return max_part + 1

part_counter = next_part_counter()
print('[resume] próximo part:', part_counter)

if not resume_manifest:
    print('[done] nenhum mês novo para processar; todos os meses selecionados já possuem Parquet.')

for item in resume_manifest:
    # Para cada mês selecionado, abre o ZIP, detecta o separador e inicia leitura incremental.
    zip_path = RAW_ZIP_DIR / Path(item['url']).name
    if not zip_path.exists():
        raise FileNotFoundError(f'ZIP não encontrado: {zip_path}. Rode a célula 6 para baixar/reutilizar o arquivo.')

    member = zip_csv_member(zip_path)
    sep = detect_sep(zip_path, member)
    has_header = csv_has_header(zip_path, member, sep)
    header = 0 if has_header else None
    names = None if has_header else CSV_COLUMNS
    print(f'[process] {zip_path.name} member={member} sep={sep!r} header={has_header}')

    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(member) as file:
            reader = pd.read_csv(
                file,
                sep=sep,
                header=header,
                names=names,
                dtype='string',
                chunksize=CHUNKSIZE,
                encoding=CSV_ENCODING,
                low_memory=False,
            )
            for chunk_index, chunk in enumerate(reader):
                original_rows = len(chunk)
                processed = transform_chunk(chunk)
                processed, removed_duplicates = deduplicate_chunk(processed)
                update_metrics(processed, original_rows, removed_duplicates)

                if schema_columns is None:
                    schema_columns = [{'name': c, 'pandas_dtype': str(t)} for c, t in processed.dtypes.items()]
                if processed.empty:
                    continue

                year = int(processed['ano_vacinacao'].dropna().iloc[0]) if processed['ano_vacinacao'].notna().any() else item['year']
                month = int(processed['mes_vacinacao'].dropna().iloc[0]) if processed['mes_vacinacao'].notna().any() else item['month']

                out_dir = PROCESSED_DIR / f'year={year}' / f'month={month:02d}'
                out_dir.mkdir(parents=True, exist_ok=True)
                out_path = out_dir / f'part-{part_counter:06d}.parquet'
                processed.to_parquet(out_path, index=False)
                part_counter += 1

                # Libera referências grandes o quanto antes para ajudar o Colab.
                del chunk, processed

                if chunk_index % 10 == 0:
                    print(f'  chunk={chunk_index} saved={out_path.name}')

print('[done] resume parquet parts até:', part_counter - 1)
print('[done] métricas desta retomada:')
display(pd.DataFrame(
    [{'metric': k, 'value': v, 'description': k.replace('_', ' ')} for k, v in {**metrics, **missing_metrics}.items()]
))


In [ ]:
# Célula 8 - Gera artefatos finais: agregados, documentação e banco DuckDB.
# A saída desta célula é o pacote consumível do projeto: CSVs analíticos,
# relatórios de qualidade, dicionário de dados, metadados e arquivo .duckdb.

# Esta célula usa DuckDB para gerar os agregados diretamente dos Parquets.
# Evitamos carregar todos os Parquets em Pandas, porque a execução anual tem
# mais de 170 milhões de registros e pode estourar a RAM do Colab.

# Schema observado na base curada, útil para auditoria e para o data paper.
schema = {
    'dataset': 'VacinaBR-PNI',
    'source': 'Portal de Dados Abertos do SUS - CSVs mensais',
    'columns': schema_columns or [],
}
with (DOCS_DIR / 'schema.json').open('w', encoding='utf-8') as file:
    json.dump(schema, file, ensure_ascii=False, indent=2)

# Dicionário de dados dos principais campos brutos preservados e campos derivados.
data_dictionary = [
    ('data_vacina', 'date', 'Data de aplicacao da vacina.'),
    ('ano_vacinacao', 'integer', 'Ano extraido da data de vacinacao.'),
    ('mes_vacinacao', 'integer', 'Mes extraido da data de vacinacao.'),
    ('trimestre_vacinacao', 'string', 'Trimestre da vacinacao.'),
    ('semana_epidemiologica', 'integer', 'Semana epidemiologica estimada a partir da data de vacinacao.'),
    ('sexo_paciente', 'string', 'Sexo informado do paciente.'),
    ('numero_idade_paciente', 'number', 'Idade informada do paciente.'),
    ('faixa_etaria', 'string', 'Faixa etaria derivada da idade.'),
    ('idade_valida', 'boolean', 'Indica se a idade esta no intervalo esperado de 0 a 130 anos.'),
    ('uf_paciente', 'string', 'Unidade federativa do paciente.'),
    ('regiao_paciente', 'string', 'Regiao geografica derivada da UF do paciente.'),
    ('codigo_municipio_paciente', 'string', 'Codigo do municipio do paciente.'),
    ('nome_municipio_paciente', 'string', 'Nome do municipio do paciente.'),
    ('uf_estabelecimento', 'string', 'Unidade federativa do estabelecimento.'),
    ('regiao_estabelecimento', 'string', 'Regiao geografica derivada da UF do estabelecimento.'),
    ('codigo_municipio_estabelecimento', 'string', 'Codigo do municipio do estabelecimento.'),
    ('municipio_paciente_igual_estabelecimento', 'boolean', 'Indica se paciente e estabelecimento pertencem ao mesmo municipio.'),
    ('codigo_vacina', 'string', 'Codigo da vacina.'),
    ('sg_vacina', 'string', 'Sigla da vacina ou imunobiologico.'),
    ('descricao_vacina', 'string', 'Nome ou descricao da vacina ou imunobiologico.'),
    ('descricao_dose_vacina', 'string', 'Descricao da dose aplicada.'),
    ('descricao_vacina_fabricante', 'string', 'Fabricante informado da vacina.'),
    ('descricao_estrategia_vacinacao', 'string', 'Estrategia de vacinacao informada.'),
    ('descricao_sistema_origem', 'string', 'Sistema de origem do registro.'),
    ('st_documento', 'string', 'Status documental do registro.'),
    ('registro_deletado_rnds', 'boolean', 'Indica se o registro possui data de delecao na RNDS.'),
    ('documento_final', 'boolean', 'Indica se o status do documento e final.'),
    ('registro_valido_documento', 'boolean', 'Indica documento final e nao deletado.'),
    ('registro_completo', 'boolean', 'Indica preenchimento dos campos essenciais.'),
]
data_dictionary_df = pd.DataFrame(data_dictionary, columns=['field', 'type', 'description'])
derived_fields = {'ano_vacinacao', 'mes_vacinacao', 'trimestre_vacinacao', 'semana_epidemiologica', 'faixa_etaria', 'regiao_paciente', 'regiao_estabelecimento', 'municipio_paciente_igual_estabelecimento'}
quality_fields = {'idade_valida', 'registro_deletado_rnds', 'documento_final', 'registro_valido_documento', 'registro_completo'}
example_values = {
    'data_vacina': '2025-01-14',
    'uf_paciente': 'SP',
    'regiao_paciente': 'Sudeste',
    'nome_municipio_paciente': 'SAO PAULO',
    'sg_vacina': 'IGHT',
    'descricao_vacina': 'Imunoglobulina humana antitetano',
    'descricao_vacina_fabricante': 'CSL BEHRING',
}
data_dictionary_df['source_or_derived'] = data_dictionary_df['field'].map(lambda field: 'derived_quality' if field in quality_fields else ('derived' if field in derived_fields else 'source'))
data_dictionary_df['example_value'] = data_dictionary_df['field'].map(example_values).fillna('')
data_dictionary_df['sensitive'] = data_dictionary_df['field'].isin(SENSITIVE_COLUMNS)
data_dictionary_df.to_csv(DOCS_DIR / 'data_dictionary.csv', index=False, encoding='utf-8')

# Metadados de proveniência: fonte oficial, recursos usados, locais de armazenamento
# e descrição da camada DuckDB usada para consultas SQL.
source_metadata = {
    'dataset': 'VacinaBR-PNI',
    'source_page': manifest_df['source_page'].iloc[0],
    'resources': MANIFEST,
    'raw_storage': str(RAW_ZIP_DIR),
    'processed_storage': str(PROCESSED_DIR),
    'database_layer': {
        'database': 'DuckDB',
        'database_path': str(DUCKDB_PATH),
        'purpose': 'Consultar os Parquets curados e materializar agregados analiticos por SQL.',
    },
    'sensitive_fields_removed_from_processed': SENSITIVE_COLUMNS,
}
with (DOCS_DIR / 'source_metadata.json').open('w', encoding='utf-8') as file:
    json.dump(source_metadata, file, ensure_ascii=False, indent=2)

# DuckDB: cria uma view SQL sobre todos os Parquets particionados.
# A view evita duplicar os dados curados e permite consultar o dataset por SQL.
parquet_pattern = str(PROCESSED_DIR / 'year=*' / 'month=*' / '*.parquet')
con = duckdb.connect(str(DUCKDB_PATH))
con.execute(f"""
CREATE OR REPLACE VIEW vacinacao_curada AS
SELECT *
FROM read_parquet('{parquet_pattern}', union_by_name = true)
""")
con.execute("""
CREATE OR REPLACE TABLE dataset_metadata AS
SELECT
    'VacinaBR-PNI' AS dataset,
    'Portal de Dados Abertos do SUS - CSVs mensais' AS source,
    current_timestamp AS generated_at,
    (SELECT count(*) FROM vacinacao_curada) AS total_records
""")

# Relatório de validação: recalculado diretamente dos Parquets via DuckDB.
# Isso torna a célula 8 independente das variáveis em memória da célula 7.
def quote_identifier(name: str) -> str:
    return chr(34) + name.replace(chr(34), chr(34) + chr(34)) + chr(34)

available_columns = [
    row[0]
    for row in con.execute('DESCRIBE vacinacao_curada').fetchall()
]
quoted_columns = [quote_identifier(col) for col in available_columns]
record_count = int(con.execute('SELECT count(*) FROM vacinacao_curada').fetchone()[0])

if quoted_columns:
    # Calcula o total de células em Python para evitar overflow INT32 no DuckDB
    # em execuções anuais com centenas de milhões de registros.
    total_cells = record_count * len(quoted_columns)
    count_present_expr = ' + '.join(f'count({col})' for col in quoted_columns)
    present_cells = int(
        con.execute(f'SELECT {count_present_expr} FROM vacinacao_curada').fetchone()[0]
    )
    total_missing_values = total_cells - present_cells
else:
    total_missing_values = 0

def count_condition(condition: str) -> int:
    return int(con.execute(f'SELECT count(*) FROM vacinacao_curada WHERE {condition}').fetchone()[0])

def missing_count(column: str) -> int:
    if column not in available_columns:
        return 0
    col = quote_identifier(column)
    return int(con.execute(f'SELECT count(*) - count({col}) FROM vacinacao_curada').fetchone()[0])

def false_or_null_count(column: str) -> int:
    if column not in available_columns:
        return 0
    col = quote_identifier(column)
    return count_condition(f'{col} IS NULL OR {col} = false')

def true_count(column: str) -> int:
    if column not in available_columns:
        return 0
    col = quote_identifier(column)
    return count_condition(f'{col} = true')

validation_values = {
    'original_records': record_count,
    'processed_records': record_count,
    'removed_duplicate_records': 0,
    'total_missing_values': total_missing_values,
    'invalid_age_records': false_or_null_count('idade_valida'),
    'invalid_date_records': missing_count('data_vacina'),
    'complete_records': true_count('registro_completo'),
    'incomplete_records': false_or_null_count('registro_completo'),
    'valid_document_records': true_count('registro_valido_documento'),
    'invalid_document_records': false_or_null_count('registro_valido_documento'),
}

for column in [
    'data_vacina',
    'sexo_paciente',
    'numero_idade_paciente',
    'uf_paciente',
    'codigo_municipio_paciente',
    'sg_vacina',
    'descricao_vacina_fabricante',
]:
    validation_values[f'missing_{column}'] = missing_count(column)

validation_rows = [
    {'metric': key, 'value': value, 'description': key.replace('_', ' ')}
    for key, value in validation_values.items()
]
pd.DataFrame(validation_rows).to_csv(DOCS_DIR / 'validation_report.csv', index=False, encoding='utf-8')

# Consultas SQL que materializam tabelas de resumo dentro do arquivo .duckdb.
duckdb_summary_queries = {
    'monthly_vaccination_summary': """
        SELECT ano_vacinacao, mes_vacinacao, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY ano_vacinacao, mes_vacinacao
        ORDER BY ano_vacinacao, mes_vacinacao
    """,
    'state_vaccination_summary': """
        SELECT uf_paciente, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY uf_paciente
        ORDER BY doses_aplicadas DESC
    """,
    'region_vaccination_summary': """
        SELECT regiao_paciente, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY regiao_paciente
        ORDER BY doses_aplicadas DESC
    """,
    'vaccine_type_summary': """
        SELECT sg_vacina, descricao_vacina, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY sg_vacina, descricao_vacina
        ORDER BY doses_aplicadas DESC
    """,
    'manufacturer_summary': """
        SELECT descricao_vacina_fabricante, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY descricao_vacina_fabricante
        ORDER BY doses_aplicadas DESC
    """,
    'age_group_summary': """
        SELECT faixa_etaria, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY faixa_etaria
        ORDER BY doses_aplicadas DESC
    """,
    'sex_summary': """
        SELECT sexo_paciente, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY sexo_paciente
        ORDER BY doses_aplicadas DESC
    """,
    'monthly_vaccine_type_summary': """
        SELECT ano_vacinacao, mes_vacinacao, sg_vacina, descricao_vacina, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY ano_vacinacao, mes_vacinacao, sg_vacina, descricao_vacina
        ORDER BY ano_vacinacao, mes_vacinacao, doses_aplicadas DESC
    """,
    'state_vaccine_type_summary': """
        SELECT uf_paciente, sg_vacina, descricao_vacina, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY uf_paciente, sg_vacina, descricao_vacina
        ORDER BY uf_paciente, doses_aplicadas DESC
    """,
    'municipality_vaccination_summary': """
        SELECT uf_paciente, regiao_paciente, codigo_municipio_paciente, nome_municipio_paciente, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY uf_paciente, regiao_paciente, codigo_municipio_paciente, nome_municipio_paciente
        ORDER BY doses_aplicadas DESC
    """,
    'state_municipality_vaccine_summary': """
        SELECT uf_paciente, codigo_municipio_paciente, nome_municipio_paciente, sg_vacina, descricao_vacina, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY uf_paciente, codigo_municipio_paciente, nome_municipio_paciente, sg_vacina, descricao_vacina
        ORDER BY uf_paciente, nome_municipio_paciente, doses_aplicadas DESC
    """,
    'quality_by_month': """
        SELECT ano_vacinacao, mes_vacinacao, count(*) AS total_registros,
               sum(CASE WHEN registro_completo THEN 1 ELSE 0 END) AS registros_completos,
               sum(CASE WHEN idade_valida THEN 1 ELSE 0 END) AS idades_validas,
               sum(CASE WHEN registro_valido_documento THEN 1 ELSE 0 END) AS documentos_validos,
               round(100.0 * sum(CASE WHEN registro_completo THEN 1 ELSE 0 END) / count(*), 2) AS pct_completude,
               round(100.0 * sum(CASE WHEN idade_valida THEN 1 ELSE 0 END) / count(*), 2) AS pct_idade_valida,
               round(100.0 * sum(CASE WHEN registro_valido_documento THEN 1 ELSE 0 END) / count(*), 2) AS pct_documento_valido
        FROM vacinacao_curada
        GROUP BY ano_vacinacao, mes_vacinacao
        ORDER BY ano_vacinacao, mes_vacinacao
    """,
    'quality_by_state': """
        SELECT uf_paciente, count(*) AS total_registros,
               sum(CASE WHEN registro_completo THEN 1 ELSE 0 END) AS registros_completos,
               sum(CASE WHEN idade_valida THEN 1 ELSE 0 END) AS idades_validas,
               sum(CASE WHEN registro_valido_documento THEN 1 ELSE 0 END) AS documentos_validos,
               round(100.0 * sum(CASE WHEN registro_completo THEN 1 ELSE 0 END) / count(*), 2) AS pct_completude,
               round(100.0 * sum(CASE WHEN idade_valida THEN 1 ELSE 0 END) / count(*), 2) AS pct_idade_valida,
               round(100.0 * sum(CASE WHEN registro_valido_documento THEN 1 ELSE 0 END) / count(*), 2) AS pct_documento_valido
        FROM vacinacao_curada
        GROUP BY uf_paciente
        ORDER BY total_registros DESC
    """,
    'quality_by_vaccine': """
        SELECT sg_vacina, descricao_vacina, count(*) AS total_registros,
               sum(CASE WHEN registro_completo THEN 1 ELSE 0 END) AS registros_completos,
               sum(CASE WHEN idade_valida THEN 1 ELSE 0 END) AS idades_validas,
               sum(CASE WHEN registro_valido_documento THEN 1 ELSE 0 END) AS documentos_validos,
               round(100.0 * sum(CASE WHEN registro_completo THEN 1 ELSE 0 END) / count(*), 2) AS pct_completude,
               round(100.0 * sum(CASE WHEN idade_valida THEN 1 ELSE 0 END) / count(*), 2) AS pct_idade_valida,
               round(100.0 * sum(CASE WHEN registro_valido_documento THEN 1 ELSE 0 END) / count(*), 2) AS pct_documento_valido
        FROM vacinacao_curada
        GROUP BY sg_vacina, descricao_vacina
        ORDER BY total_registros DESC
    """,
    'vaccine_dictionary': """
        SELECT codigo_vacina, sg_vacina, descricao_vacina, count(*) AS registros_observados
        FROM vacinacao_curada
        GROUP BY codigo_vacina, sg_vacina, descricao_vacina
        ORDER BY sg_vacina, descricao_vacina, codigo_vacina
    """,
}
for table_name, query in duckdb_summary_queries.items():
    # Cada tabela é recriada para refletir a versão mais recente dos Parquets.
    print('[duckdb] materializando', table_name)
    con.execute(f'CREATE OR REPLACE TABLE {table_name} AS {query}')

# Exporta os agregados para CSV sem carregar a base completa em Pandas.
for table_name in duckdb_summary_queries:
    output_csv = (ANALYTICS_DIR / f'{table_name}.csv').as_posix()
    print('[csv] exportando', output_csv)
    con.execute(f"COPY {table_name} TO '{output_csv}' (HEADER, DELIMITER ',')")

# Catálogo simples para documentar quais objetos existem no DuckDB final.
duckdb_catalog = {
    'database': str(DUCKDB_PATH),
    'source_parquet_pattern': parquet_pattern,
    'view': 'vacinacao_curada',
    'tables': [
        row[0]
        for row in con.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
            ORDER BY table_name
        """).fetchall()
    ],
}
with (DOCS_DIR / 'duckdb_catalog.json').open('w', encoding='utf-8') as file:
    json.dump(duckdb_catalog, file, ensure_ascii=False, indent=2)

# Prévia final para conferir rapidamente se o banco e os agregados foram gerados.
print('[done] DuckDB:', DUCKDB_PATH)
display(con.sql('SELECT * FROM dataset_metadata').df())
display(con.sql('SELECT * FROM state_vaccination_summary LIMIT 10').df())
con.close()

print('[done] Analytics:', ANALYTICS_DIR)
print('[done] Docs:', DOCS_DIR)
pd.read_csv(DOCS_DIR / 'validation_report.csv').head(20)

## Fechamento da execução completa

As células abaixo consolidam os números necessários para o artigo, para o mapa e para o pacote de publicação. Rode-as depois que todos os meses forem processados e depois que os CSVs analíticos e o DuckDB forem gerados.

In [ ]:
# Célula final A - Valida consistência da run completa e imprime os números
# que devem ir para o artigo e para a documentação do dataset.
from pathlib import Path

def file_size_mb(path: Path) -> float:
    return round(path.stat().st_size / (1024 * 1024), 2) if path.exists() else 0.0

def dir_size_mb(path: Path) -> float:
    if not path.exists():
        return 0.0
    return round(sum(p.stat().st_size for p in path.rglob('*') if p.is_file()) / (1024 * 1024), 2)

def metric_value(validation_df: pd.DataFrame, metric: str, default=None) -> int:
    value = validation_df.loc[validation_df['metric'] == metric, 'value']
    if value.empty:
        if default is None:
            raise ValueError(f'Métrica não encontrada em validation_report.csv: {metric}')
        print(f'[warn] Métrica ausente em validation_report.csv: {metric}; usando valor reconstruído/default={default}')
        return int(default)
    return int(value.iloc[0])

def missing_state_records_from_quality(quality_state_df: pd.DataFrame) -> int:
    if 'uf_paciente' not in quality_state_df.columns or 'total_registros' not in quality_state_df.columns:
        return 0
    uf = quality_state_df['uf_paciente'].astype('string').str.strip()
    return int(quality_state_df.loc[uf.isna() | (uf == ''), 'total_registros'].sum())

def missing_municipality_records_from_summary(municipality_df: pd.DataFrame) -> int:
    if 'codigo_municipio_paciente' not in municipality_df.columns or 'doses_aplicadas' not in municipality_df.columns:
        return 0
    code = municipality_df['codigo_municipio_paciente'].astype('string').str.strip()
    return int(municipality_df.loc[code.isna() | (code == ''), 'doses_aplicadas'].sum())

validation = pd.read_csv(DOCS_DIR / 'validation_report.csv')
monthly = pd.read_csv(ANALYTICS_DIR / 'monthly_vaccination_summary.csv')
vaccine = pd.read_csv(ANALYTICS_DIR / 'vaccine_type_summary.csv')
municipality = pd.read_csv(ANALYTICS_DIR / 'municipality_vaccination_summary.csv')
quality_state = pd.read_csv(ANALYTICS_DIR / 'quality_by_state.csv')

processed_records = metric_value(validation, 'processed_records')
original_records = metric_value(validation, 'original_records')
removed_duplicates = metric_value(validation, 'removed_duplicate_records')
missing_values = metric_value(validation, 'total_missing_values')
complete_records = metric_value(validation, 'complete_records')
incomplete_records = metric_value(validation, 'incomplete_records')
valid_document_records = metric_value(validation, 'valid_document_records')
invalid_age_records = metric_value(validation, 'invalid_age_records')
invalid_date_records = metric_value(validation, 'invalid_date_records')
missing_uf = metric_value(
    validation,
    'missing_uf_paciente',
    default=missing_state_records_from_quality(quality_state),
)
missing_municipality = metric_value(
    validation,
    'missing_codigo_municipio_paciente',
    default=missing_municipality_records_from_summary(municipality),
)

monthly_total = int(monthly['doses_aplicadas'].sum())
vaccine_total = int(vaccine['doses_aplicadas'].sum())
municipality_total = int(municipality['doses_aplicadas'].sum())
parquet_files = list(PROCESSED_DIR.rglob('*.parquet'))

checks = {
    'monthly_total_equals_processed': monthly_total == processed_records,
    'vaccine_total_equals_processed': vaccine_total == processed_records,
    'municipality_total_equals_processed': municipality_total == processed_records,
    'valid_documents_equals_processed': valid_document_records == processed_records,
}

run_summary = pd.DataFrame([
    {'metric': 'original_records', 'value': original_records},
    {'metric': 'processed_records', 'value': processed_records},
    {'metric': 'removed_duplicate_records', 'value': removed_duplicates},
    {'metric': 'total_missing_values', 'value': missing_values},
    {'metric': 'complete_records', 'value': complete_records},
    {'metric': 'incomplete_records', 'value': incomplete_records},
    {'metric': 'valid_document_records', 'value': valid_document_records},
    {'metric': 'invalid_age_records', 'value': invalid_age_records},
    {'metric': 'invalid_date_records', 'value': invalid_date_records},
    {'metric': 'missing_uf_paciente', 'value': missing_uf},
    {'metric': 'missing_codigo_municipio_paciente', 'value': missing_municipality},
    {'metric': 'monthly_summary_total', 'value': monthly_total},
    {'metric': 'vaccine_summary_total', 'value': vaccine_total},
    {'metric': 'municipality_summary_total', 'value': municipality_total},
    {'metric': 'months_processed', 'value': int(monthly.dropna(subset=['ano_vacinacao', 'mes_vacinacao'])[['ano_vacinacao', 'mes_vacinacao']].drop_duplicates().shape[0])},
    {'metric': 'municipalities_observed', 'value': int(municipality['codigo_municipio_paciente'].dropna().astype(str).nunique())},
    {'metric': 'distinct_sg_vacina', 'value': int(vaccine['sg_vacina'].dropna().nunique())},
    {'metric': 'distinct_vaccine_rows', 'value': int(vaccine[['sg_vacina', 'descricao_vacina']].drop_duplicates().shape[0])},
    {'metric': 'parquet_files', 'value': len(parquet_files)},
    {'metric': 'processed_parquet_size_mb', 'value': dir_size_mb(PROCESSED_DIR)},
    {'metric': 'duckdb_size_mb', 'value': file_size_mb(DUCKDB_PATH)},
])

run_summary.to_csv(DOCS_DIR / 'run_closure_summary.csv', index=False, encoding='utf-8')
pd.DataFrame([{'check': k, 'passed': v} for k, v in checks.items()]).to_csv(
    DOCS_DIR / 'run_consistency_checks.csv', index=False, encoding='utf-8'
)

print('Resumo salvo em:', DOCS_DIR / 'run_closure_summary.csv')
print('Checks salvos em:', DOCS_DIR / 'run_consistency_checks.csv')
display(run_summary)
display(pd.DataFrame([{'check': k, 'passed': v} for k, v in checks.items()]))
display(monthly.sort_values(['ano_vacinacao', 'mes_vacinacao']))
display(quality_state.sort_values('pct_completude').head(10))

failed_checks = [name for name, passed in checks.items() if not passed]
if failed_checks:
    raise AssertionError(f'Falha nos checks de consistência: {failed_checks}')

In [ ]:
# Célula final B - Gera o CSV municipal mapeável e o mapa Plotly da run completa.
# O mapa usa apenas registros com código municipal compatível com o IBGE.
import json

FIGURES_DIR = DRIVE_ROOT / 'paper' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

IBGE_MUNICIPALITIES_CSV = DRIVE_ROOT / 'data_sources' / 'ibge_municipalities.csv'
if not IBGE_MUNICIPALITIES_CSV.exists():
    raise FileNotFoundError(
        'Arquivo IBGE não encontrado. Copie data_sources/ibge_municipalities.csv '
        f'para {IBGE_MUNICIPALITIES_CSV} antes de gerar o mapa.'
    )

municipality = pd.read_csv(ANALYTICS_DIR / 'municipality_vaccination_summary.csv')
geo = pd.read_csv(IBGE_MUNICIPALITIES_CSV)

municipality['codigo_municipio_paciente'] = (
    municipality['codigo_municipio_paciente']
    .astype('string')
    .str.replace(r'\.0$', '', regex=True)
    .str.zfill(6)
)
geo['codigo_municipio'] = (
    geo['codigo_municipio']
    .astype('string')
    .str.replace(r'\.0$', '', regex=True)
    .str.zfill(6)
)

mapped = municipality.merge(
    geo[['codigo_municipio', 'nome_municipio_ibge', 'uf', 'nome_uf', 'regiao', 'nome_regiao', 'latitude', 'longitude']],
    left_on='codigo_municipio_paciente',
    right_on='codigo_municipio',
    how='left',
)
mapped['doses_aplicadas'] = pd.to_numeric(mapped['doses_aplicadas'], errors='coerce')
mapped = mapped.dropna(subset=['latitude', 'longitude', 'doses_aplicadas']).copy()
mapped['nome_municipio_mapa'] = mapped['nome_municipio_ibge'].fillna(mapped['nome_municipio_paciente'])

mapped = (
    mapped.groupby(
        ['codigo_municipio_paciente', 'nome_municipio_mapa', 'uf_paciente', 'regiao_paciente', 'latitude', 'longitude'],
        dropna=False,
        as_index=False,
    )['doses_aplicadas']
    .sum()
    .sort_values('doses_aplicadas', ascending=False)
)

mappable_total = int(mapped['doses_aplicadas'].sum())
processed_records = metric_value(pd.read_csv(DOCS_DIR / 'validation_report.csv'), 'processed_records')
not_mappable_total = processed_records - mappable_total

mapped.to_csv(ANALYTICS_DIR / 'municipality_vaccination_summary_mappable.csv', index=False, encoding='utf-8')
pd.DataFrame([
    {'metric': 'mappable_municipalities', 'value': int(mapped.shape[0])},
    {'metric': 'mappable_doses', 'value': mappable_total},
    {'metric': 'not_mappable_doses', 'value': not_mappable_total},
    {'metric': 'pct_mappable_doses', 'value': round(100 * mappable_total / processed_records, 2)},
]).to_csv(DOCS_DIR / 'map_coverage_report.csv', index=False, encoding='utf-8')

plot_df = mapped.copy()
plot_df['hover'] = plot_df.apply(
    lambda row: f"{row['nome_municipio_mapa']} - {row['uf_paciente']}<br>Doses aplicadas: {int(row['doses_aplicadas']):,}".replace(',', '.'),
    axis=1,
)
max_doses = max(float(plot_df['doses_aplicadas'].max()), 1.0)
plot_df['marker_size'] = 7 + 36 * (plot_df['doses_aplicadas'] / max_doses) ** 0.5

payload = json.dumps(
    {
        'lat': plot_df['latitude'].round(6).tolist(),
        'lon': plot_df['longitude'].round(6).tolist(),
        'doses': plot_df['doses_aplicadas'].astype(int).tolist(),
        'size': plot_df['marker_size'].round(2).tolist(),
        'hover': plot_df['hover'].tolist(),
    },
    ensure_ascii=False,
)

map_html = FIGURES_DIR / 'municipality_map_plotly.html'
map_html.write_text(
    f'''<!doctype html>
<html lang="pt-BR">
<head>
  <meta charset="utf-8">
  <title>VacinaBR-PNI - Mapa Plotly</title>
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
  <style>
    body {{ margin: 0; font-family: Arial, sans-serif; background: #f8fafc; }}
    header {{ padding: 14px 18px; border-bottom: 1px solid #ddd; background: white; }}
    h1 {{ font-size: 19px; margin: 0 0 4px; }}
    p {{ margin: 0; color: #555; }}
    #map {{ width: 100vw; height: calc(100vh - 74px); }}
  </style>
</head>
<body>
  <header>
    <h1>VacinaBR-PNI: doses aplicadas por município</h1>
    <p>Mapa gerado com Plotly. Total mapeável: {len(plot_df):,} municípios e {mappable_total:,} doses.</p>
  </header>
  <div id="map"></div>
  <script>
    const data = {payload};
    const trace = {{
      type: 'scattermapbox',
      lat: data.lat,
      lon: data.lon,
      mode: 'markers',
      text: data.hover,
      hoverinfo: 'text',
      marker: {{
        size: data.size,
        color: data.doses,
        colorscale: 'Blues',
        opacity: 0.72,
        colorbar: {{ title: 'Doses' }}
      }}
    }};
    const layout = {{
      margin: {{ l: 0, r: 0, t: 0, b: 0 }},
      mapbox: {{ style: 'open-street-map', center: {{ lat: -14.2, lon: -51.9 }}, zoom: 3.2 }}
    }};
    Plotly.newPlot('map', [trace], layout, {{ responsive: true }});
  </script>
</body>
</html>
''',
    encoding='utf-8',
)

print('CSV mapeável:', ANALYTICS_DIR / 'municipality_vaccination_summary_mappable.csv')
print('Relatório do mapa:', DOCS_DIR / 'map_coverage_report.csv')
print('Mapa Plotly:', map_html)
display(pd.read_csv(DOCS_DIR / 'map_coverage_report.csv'))
display(mapped.head(10))